### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="cirrhosis_patient_survival_prediction",
    dataset_year="1984",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5R02G",
    download_description="""
We get the UCI data.

wget https://archive.ics.uci.edu/static/public/878/cirrhosis+patient+survival+prediction+dataset-1.zip && unzip cirrhosis+patient+survival+prediction+dataset-1.zip && rm cirrhosis+patient+survival+prediction+dataset-1.zip && mkdir -p local-data-warehouse/cirrhosis_patient_survival_prediction && mv cirrhosis.csv local-data-warehouse/cirrhosis_patient_survival_prediction/
""",
    # References
    academic_reference_bibtex="""@article{dickson1989prognosis,
  title={Prognosis in primary biliary cirrhosis: model for decision making},
  author={Dickson, E Rolland and Grambsch, Patricia M and Fleming, Thomas R and Fisher, Lloyd D and Langworthy, Alice},
  journal={Hepatology},
  volume={10},
  number={1},
  pages={1--7},
  year={1989},
  publisher={Wiley Online Library}
}
""",
    academic_reference_bibtex_key="dickson1989prognosis",
    license="IID",
    data_tags=["IID"],
    curation_comments="""
We start with the UCI data.

We note that this is a survival prediction task. But the way we split the data and score the task, it is not treated correctly as a survival prediction task. At the same time, it is randomized control data based on the Drug column. We treat it as a regression task to predict the time-to-death for non-censored patients, including the drug column into the modelling.

- We drop ID and Status column as they have no relevance after filtering to only non-censored patients.
- We log transform N_days and rename it to the target "log_days_to_death".
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="log_days_to_death",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "cirrhosis.csv")
print("Loaded data shape:", df.shape)

df = df[df["Status"] == "D"]
df["log_days_to_death"] = np.log(df["N_Days"])
df = df.drop(columns=["ID", "Status", "N_Days"])

as_cat_type = ["Drug", "Ascites", "Hepatomegaly", "Spiders", "Edema", "Stage", "Sex"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (418, 20)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 161
Columns: 18
Use sampling: False (sample size: 161)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Age', 'Platelets', 'Alk_Phos', 'Albumin', 'Cholesterol', 'Copper', 'SGOT', 'Tryglicerides', 'Bilirubin', 'Prothrombin']
Rows remaining as candidates after top-10 filter: 0 (of 161)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,log_days_to_death
0,Placebo,22857,F,N,N,N,N,0.7,255.0,3.74,23.0,1024.0,77.50,58.0,281.0,10.2,3.0,7.644919
1,Placebo,25329,F,N,Y,N,N,0.9,404.0,3.43,34.0,1866.0,79.05,224.0,236.0,9.9,3.0,7.487734
2,NaN,17532,F,NaN,NaN,NaN,N,2.1,NaN,4.10,NaN,NaN,NaN,NaN,200.0,9.0,3.0,6.495266
3,D-penicillamine,19270,F,N,Y,Y,N,5.0,1600.0,3.21,75.0,2656.0,82.15,174.0,181.0,10.9,3.0,7.412764
4,D-penicillamine,19540,F,N,N,N,N,0.3,233.0,4.08,20.0,622.0,66.65,68.0,358.0,9.9,3.0,7.628031


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Drug,category,36.0,22.36,2.0,"D-penicillamine, Placebo"
1,Ascites,category,36.0,22.36,2.0,"N, Y"
2,Hepatomegaly,category,36.0,22.36,2.0,"Y, N"
3,Spiders,category,36.0,22.36,2.0,"N, Y"
4,Stage,category,4.0,2.48,4.0,"4.0, 3.0, 2.0, 1.0"
5,Sex,category,0.0,0.00,2.0,"F, M"
6,Edema,category,0.0,0.00,3.0,"N, S, Y"
7,Tryglicerides,float64,48.0,29.81,90.0,"91.0, 68.0, 140.0, 88.0, 118.0, 137.0, 157.0, 63.0, 99.0, 104.0"
8,Cholesterol,float64,47.0,29.19,99.0,"260.0, 674.0, 257.0, 244.0, 178.0, 416.0, 408.0, 175.0, 932.0, 426.0"
9,Copper,float64,37.0,22.98,97.0,"58.0, 50.0, 75.0, 20.0, 52.0, 94.0, 200.0, 67.0, 161.0, 43.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Age,161.0,19694.540373,3584.646006,11273.000000,28018.000000
Bilirubin,161.0,5.539130,5.837103,0.300000,28.000000
Cholesterol,114.0,415.754386,275.030765,127.000000,1775.000000
Albumin,161.0,3.360559,0.468747,1.960000,4.520000
Copper,124.0,135.411290,98.498897,13.000000,588.000000
Alk_Phos,125.0,2594.353600,2677.108587,516.000000,13862.400000
SGOT,125.0,141.930720,58.379520,28.380000,338.000000
Tryglicerides,113.0,140.486726,79.258181,49.000000,598.000000
Platelets,155.0,242.490323,107.876154,62.000000,721.000000
Prothrombin,160.0,11.190625,1.049037,9.000000,15.200000


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column       rank                               
Ascites      1                   N    102  63.35
             2                <NA>     36  22.36
             3                   Y     23  14.29
Drug         1     D-penicillamine     65  40.37
             2             Placebo     60  37.27
             3                <NA>     36  22.36
Edema        1                   N    116  72.05
             2                   S     26  16.15
             3                   Y     19  11.80
Hepatomegaly 1                   Y     88  54.66
             2                   N     37  22.98
             3                <NA>     36  22.36
Sex          1                   F    137  85.09
             2                   M     24  14.91
Spiders      1                   N     73  45.34
             2                   Y     52  32.30
             3                <NA>     36  22.36
Stage        1                 4.0     84  52.17
             2                 3.0     48  29.81
             3                 2.0     23  14.29
             4                <NA>      4   2.48
             5                 1.0      2   1.24

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,-0.971,-1.44,1.1,0.03,log,1073.8,2.082300e+14,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits


splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to cirrhosis_patient_survival_prediction/019d5dc0-6834-7e17-89e5-c68095c2889f
019d5dc0-6834-7e17-89e5-c68095c2889f
b74d0d296ab8c6170f89dbcce47c2fddd8e89a227b863abba3711ef1ae4ca5bd
